## Design Agent

Define input and output structures
Build a prompt and a chain for recommendations on improvements
Adding in repetition for improvements.
Test on sample pain points

Input the generated panels and the pain point critiques to recommend new design suggestions

In [4]:
# Core imports
import json
import os
from typing import List, Dict, Optional, Literal

# LangChain imports
from langchain_ollama import ChatOllama
from langchain_core.prompts import (
    ChatPromptTemplate,
    FewShotPromptTemplate,
    PromptTemplate,
    MessagesPlaceholder
)
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser

# Pydantic for structured output
from pydantic import BaseModel, Field
from models.schemas import Panel, StoryboardOutput, CriticOutput

In [5]:
# Configuration
USE_REMOTE = False
#REMOTE_URL = 
#LOCAL_URL = "http://localhost:11434"
#OLLAMA_BASE_URL = REMOTE_URL if USE_REMOTE else LOCAL_URL

# Import utilities
import sys
sys.path.insert(0, '../inclass')

from llm_utils import get_llm, get_chat_model

# Get models - use qwen3:4b for reasoning tasks
model = get_llm(use_remote=USE_REMOTE, model="qwen3:4b")
chat_model = get_chat_model(use_remote=USE_REMOTE, model="qwen3:4b")

Using Ollama at http://localhost:11434
Model: qwen3:4b
Using Ollama at http://localhost:11434
Model: qwen3:4b


In [6]:
# initialize LLM
llm = ChatOllama(
    model="qwen-3:4b",
    temperature=0.7,
    #base_url=OLLAMA_BASE_URL
)

In [16]:
# load in sample data
with open('sample_critic_outputs.json', "r") as f:
    SAMPLE_CRITIC_OUTPUTS = json.load(f)

with open('sample_panel_outputs.json', "r") as f:
    SAMPLE_PANEL_OUTPUTS = json.load(f)

### Pydantic Models

In [10]:
# create design recommendation output for Design Agent
class DesignRecommendation(BaseModel):
    """Structured design recommendation output from the Design Agent"""
    panel: int = Field(..., description="The panel number that this recommendation applies to")
    pain_point: str = Field(..., description="The specific pain point identified in the panel")
    recommendation: str = Field(..., description="The design recommendation to address the pain point")

In [11]:
# create design agent ouput schema
class DesignOutput(BaseModel):
    """Sturctured output from the Design Agent"""
    recommendations: List[DesignRecommendation] = Field(..., description="A list of design recommendations to improve the storyboard")

### Design Prompt

In [14]:
design_parser = PydanticOutputParser(pydantic_object=DesignOutput)
# create design agent prompt
design_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", """You are a user experience (UX) designer tasked with improving a storyboard
          that illustrates a user's journey towards achieving a specific goal. 
         The storyboard consists of multiple panels, each depicting a step in the user's journey. 
         Your task is to analyze each panel, take the identified pain points, 
         and provide actionable design recommendations to enhance the overall user experience."""),
        ("human", """Here is the storyboard you need to suggest improvements for:
         {panels}
         These are the pain points identified in each panel:
         {pain_points}
         For each pain point, provide a specific, actionable design recommendation.
         Format your response as JSON with this structure:
         {{
            "recommendations":[
                {{
                    "panel": panel number that this recommendation applies to,
                    "pain_point": the specific pain point identified in the panel,
                    "recommendation": the design recommendation to address the pain point
                }},
                ...
            ]
         }}""")
    ])

design_chain = design_prompt | chat_model | design_parser

In [28]:
# format panels
def format_panels(panels: List[Panel]) -> str:
    return "\n".join([
        f"Panel {p.panel_number}: Action='{p.action}', Context='{p.context}', Emotion='{p.emotion}'"
        for p in panels
    ])

# format pain points
def format_pain_points(critic_output:CriticOutput) -> str:
    return "\n".join([
        f"Panel {c.panel} ({c.severity} severity): {c.pain_point} — {c.reason}"
        for c in critic_output.critiques
    ])

In [31]:
# test design agent with sample input (storyboard panel outputs and pain points)
from models.schemas import PanelCritique

test_input_critic = SAMPLE_CRITIC_OUTPUTS[0]
test_input_panels = SAMPLE_PANEL_OUTPUTS[0]

panels = [Panel(**p) for p in test_input_panels["panels"]]
critic_output = CriticOutput(
    critiques=[PanelCritique(**c) for c in test_input_critic["critiques"]]
)

# format panels and pain points for input into design agent
formatted_panels = format_panels(panels)
formatted_pain_points = format_pain_points(critic_output)

result = design_chain.invoke({"panels": formatted_panels, "pain_points": formatted_pain_points})
print("Design Agent Recommendations:")
print(result)



Design Agent Recommendations:
recommendations=[DesignRecommendation(panel=1, pain_point='No urgency indicators on the homepage for last-minute bookings — Users under time pressure need critical actions surfaced immediately. The homepage does not prioritize urgent booking flows, increasing cognitive load.', recommendation="Implement a dedicated 'Last Minute Bookings' section on homepage with a prominent countdown timer (e.g., '2 hours until departure') and a 'Book Now' CTA that auto-suggests time-sensitive filters (e.g., departure within next 2 hours)."), DesignRecommendation(panel=2, pain_point="Search results load slowly, increasing user stress — Slow feedback violates Nielsen's visibility of system status heuristic. Users under time pressure will abandon if they cannot see progress.", recommendation="Add real-time loading indicators with estimated completion time (e.g., 'Loading results in 3 seconds') and a progress bar that updates during search, avoiding spinner-only feedback to re

In [36]:
for r in result.recommendations:
    print(f"Panel Number: {r.panel}")
    print(f"Pain Point: {r.pain_point}")
    print(f"Recommnedation: {r.recommendation}")
    print("-" * 40)

Panel Number: 1
Pain Point: No urgency indicators on the homepage for last-minute bookings — Users under time pressure need critical actions surfaced immediately. The homepage does not prioritize urgent booking flows, increasing cognitive load.
Recommnedation: Implement a dedicated 'Last Minute Bookings' section on homepage with a prominent countdown timer (e.g., '2 hours until departure') and a 'Book Now' CTA that auto-suggests time-sensitive filters (e.g., departure within next 2 hours).
----------------------------------------
Panel Number: 2
Pain Point: Search results load slowly, increasing user stress — Slow feedback violates Nielsen's visibility of system status heuristic. Users under time pressure will abandon if they cannot see progress.
Recommnedation: Add real-time loading indicators with estimated completion time (e.g., 'Loading results in 3 seconds') and a progress bar that updates during search, avoiding spinner-only feedback to reduce perceived wait time.
---------------